# Notebook 07 — SWaT: MAML-AE against baselines within one plant

*Corrected version of the original `07-swat-maml-clean (1).ipynb`.*

**What this notebook does.** For each training seed it meta-trains MAML on the SWaT
meta-training regimes, trains the baselines, and scores everything on the SWaT attack
recording at 20, 50 and 100 support windows. It then summarises across training seeds.

**Why.** The original ran one training run and compared MAML with baselines that had seen
more data. Its "MAML below Static" (SWaT) and "MAML about equal to Static" (WADI) findings
need a fair and repeated test.

**Input.** The output of Notebook_03 (`swat_clean.npz`, `swat_regimes.npz`,
`plant_data_summary.json`) and this repository folder, found automatically.

**Output.** `swat_within_seed<seed>.json` per training seed, `swat_within_summary.json`,
and model checkpoints. Expected time on a Kaggle GPU: about 2.5 hours per training seed; the
run resumes after a timeout (attach the previous output and commit again).

### What was corrected
1. **Same training data for MAML and its main comparison.** The original MAML saw only the
   meta-training regimes, while Static-AE and MLP-AE were trained on all normal windows,
   including the validation and test regimes. The main Static-AE and MLP-AE now train on
   the same meta-training windows as MAML. The original "all normal data" versions are kept,
   labelled legacy, so the old numbers can be lined up.
2. **Five independent training seeds.** The original trained once; its five "seeds" only
   changed the support draw. Support draws now use separate seeds, shared by all methods.
3. **Isolation Forest is given the same K support windows** as the other methods. The
   original fitted it on 2,000 normal windows at every shot count; that version is kept as
   legacy.
4. **Zero-step controls** for MAML and Static, and an **untrained LSTM-AE floor**.
5. **Label-free F1** (threshold = mean + 2 SD of the support errors) is reported next to the
   oracle best-F1, which uses the labels. The original reported only the oracle value.
6. **Stricter label rule** reported as a sensitivity check: a window counts as anomalous only
   if at least half its steps are attacks (the original rule: at least one step).
7. **Held-out-regime test.** The original defined test regimes but never used them. Here
   the model also adapts on normal windows from a regime it never saw in training and is
   scored on the attack windows of that regime, where at least 20 anomalous and 20 normal
   windows exist.
8. **Validation uses 16 fixed episodes** with their own seed (the single validation regime
   gave one random episode per check in the original).
9. Every result file records configuration, seeds, code version and time.

**Kept as in the original:** cleaning, windows and scaling (Notebook_03); FOMAML with inner
SGD 0.01 x 10 steps, Adam 0.001, 4 tasks, 20/20 support/query, clipping 1.0, up to
30,000 outer steps, validation every 500 steps with the learning-rate halving schedule,
best checkpoint; evaluation adapts 10 steps at 0.01; the conventional-training recipe.

In [ ]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import json, time, numpy as np, torch
from sklearn.ensemble import IsolationForest
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| SMOKE TEST (numbers are meaningless)" if SMOKE else "")

## 1 — Configuration

In [ ]:
PLANT = "swat"
CFG = dict(train_seeds=[42, 123, 456, 789, 1024], support_seeds=[0, 1, 2, 3, 4], k_shots=[20, 50, 100],
           n_outer=30000, val_every=500, inner_lr=0.01, inner_steps=10, outer_lr=1e-3,
           tasks_per_batch=4, support_size=20, query_size=20, use_scheduler=True,
           val_episodes_per_task=16, val_seed=2024, adapt_steps=10, adapt_lr=0.01,
           static_max_epochs=150, static_patience=15, iforest_legacy_windows=2000,
           min_windows_per_class_heldout=20, tost_margin=0.02)
if SMOKE:
    CFG.update(train_seeds=[42], support_seeds=[0], k_shots=[20], n_outer=4, val_every=2,
               val_episodes_per_task=1, static_max_epochs=1, static_patience=1, iforest_legacy_windows=100)
TRAIN_SEEDS = CFG["train_seeds"]      # edit to split the work across Kaggle sessions
print(CFG)

## 2 — Data and regimes

In [ ]:
D = sc.load_plant(PLANT)
sp = D["split"]
train_windows = {g: D["normal"][D["normal_regime"] == g] for g in sp["meta_train"]}
val_episodes = sc.fixed_episodes({g: D["normal"][D["normal_regime"] == g] for g in sp["meta_val"]},
                                 CFG["val_episodes_per_task"], CFG["val_seed"],
                                 CFG["support_size"], CFG["query_size"])
pool_same = np.concatenate(list(train_windows.values()))
pool_all = D["normal"]
print(f"normal windows {len(D['normal'])}, attack windows {len(D['attack'])}, anomalous (any) {D['any'].sum()}, (half) {D['half'].sum()}")
print(f"regimes: train {sp['meta_train']} ({len(pool_same)} windows), val {sp['meta_val']}, test {sp['meta_test']}")
heldout = {}
for g in sp["meta_test"]:
    m = D["attack_regime"] == g
    n_anom, n_norm = int(D["any"][m].sum()), int((D["any"][m] == 0).sum())
    ok = n_anom >= CFG["min_windows_per_class_heldout"] and n_norm >= CFG["min_windows_per_class_heldout"]
    heldout[g] = {"mask": m, "pool": D["normal"][D["normal_regime"] == g], "testable": ok,
                  "anomalous": n_anom, "normal": n_norm}
    print(f"  test regime {g}: {len(heldout[g]['pool'])} normal windows; attack windows {int(m.sum())} "
          f"(anomalous {n_anom}, normal {n_norm}) -> {'testable' if ok else 'NOT testable (fewer than 20 of a class)'}")

## 3 — Finding earlier results and checkpoints (for resuming)

In [ ]:
def find_all(name):
    hits = [os.path.join(OUT, name)] if os.path.exists(os.path.join(OUT, name)) else []
    if os.path.isdir("/kaggle/input"):
        hits += [os.path.join(d, name) for d, _, f in os.walk("/kaggle/input") if name in f]
    return hits

def latest_checkpoint(name):
    best, best_step = None, -1
    for p in find_all(name):
        try:
            s = torch.load(p, map_location="cpu", weights_only=False).get("step", 0)
        except Exception:
            continue
        if s > best_step:
            best, best_step = p, s
    return best

def result_name(seed):
    return f"{PLANT}_within_seed{seed}{'_SMOKE' if SMOKE else ''}.json"

## 4 — Training for one seed

MAML, Static-AE and MLP-AE learn from the meta-training regimes only. The legacy
Static-AE and MLP-AE learn from all normal windows, as in the original.

In [ ]:
def train_conv(key, cls, pool, seed):
    fname = f"{PLANT}_{key}_seed{seed}.pt"
    sc.seed_everything(seed)
    m = cls(D["normal"].shape[2]).to(DEVICE)
    hit = find_all(fname)
    if hit:
        ck = torch.load(hit[0], map_location=DEVICE, weights_only=False)
        m.load_state_dict(ck["model_state_dict"]); return m, ck["info"]
    m, info = sc.train_conventional(m, pool, DEVICE, seed=seed, max_epochs=CFG["static_max_epochs"],
                                    patience=CFG["static_patience"])
    torch.save({"model_state_dict": m.state_dict(), "info": info}, os.path.join(OUT, fname))
    return m, info

def train_models(seed):
    models, info = {}, {}
    name = f"{PLANT}_maml_seed{seed}"
    sc.seed_everything(seed)
    maml = sc.LSTMAutoencoder(D["normal"].shape[2]).to(DEVICE)
    done = find_all(f"{name}_best.pt")
    if done:
        ck = torch.load(done[0], map_location=DEVICE, weights_only=False)
        maml.load_state_dict(ck["model_state_dict"]); info["maml"] = ck["info"]
    else:
        maml, info["maml"] = sc.train_maml(
            maml, train_windows, val_episodes, DEVICE, seed=seed, n_outer=CFG["n_outer"],
            val_every=CFG["val_every"], inner_lr=CFG["inner_lr"], inner_steps=CFG["inner_steps"],
            outer_lr=CFG["outer_lr"], tasks_per_batch=min(CFG["tasks_per_batch"], len(train_windows)),
            support_size=CFG["support_size"], query_size=CFG["query_size"],
            use_scheduler=CFG["use_scheduler"], ckpt_path=os.path.join(OUT, f"{name}_ckpt.pt"),
            resume_from=latest_checkpoint(f"{name}_ckpt.pt"))
        torch.save({"model_state_dict": maml.state_dict(), "info": info["maml"]}, os.path.join(OUT, f"{name}_best.pt"))
    models["maml"] = maml
    for key, cls, pool in [("static", sc.LSTMAutoencoder, pool_same), ("mlp", sc.MLPAutoencoder, pool_same),
                           ("static_all_legacy", sc.LSTMAutoencoder, pool_all),
                           ("mlp_all_legacy", sc.MLPAutoencoder, pool_all)]:
        models[key], info[key] = train_conv(key, cls, pool, seed)
    return models, info

## 5 — Scoring for one seed

**Whole-plant view** (the original set-up, a sanity baseline rather than a few-shot test):
support windows are drawn from all normal windows and every attack-recording window is
scored. **Held-out-regime view** (the few-shot test): support windows come from a test
regime never used in training, and only that regime's attack-recording windows are scored.

In [ ]:
def metrics_both(errs, tau, mask=None):
    y_any, y_half = D["any"], D["half"]
    if mask is not None:
        errs_, y_any, y_half = errs[mask], y_any[mask], y_half[mask]
    else:
        errs_ = errs
    return {"any": sc.detection_metrics(errs_, y_any, tau), "half": sc.detection_metrics(errs_, y_half, tau)}

def adapted_scores(model, sup, steps):
    a = sc.inner_adapt(model, sc.to_tensor(sup, DEVICE), CFG["adapt_lr"], steps) if steps else model
    return sc.window_errors(a, D["attack"], DEVICE), sc.label_free_threshold(sc.window_errors(a, sup, DEVICE)), a

def iforest(fit_windows, seed):
    iso = IsolationForest(n_estimators=100, contamination="auto", random_state=seed)
    iso.fit(fit_windows.reshape(len(fit_windows), -1))
    s = -iso.score_samples(D["attack"].reshape(len(D["attack"]), -1))
    return s, sc.label_free_threshold(-iso.score_samples(fit_windows.reshape(len(fit_windows), -1)))

def score_all(models, sup, s, seed, mask=None, legacy=True):
    r, adapt = {}, {}
    for name, key, steps in [("MAML-AE", "maml", CFG["adapt_steps"]), ("MAML-AE (0 steps)", "maml", 0),
                             ("Static-AE", "static", CFG["adapt_steps"]), ("Static-AE (0 steps)", "static", 0),
                             ("MLP-AE", "mlp", CFG["adapt_steps"])]:
        e, tau, a = adapted_scores(models[key], sup, steps)
        r[name] = metrics_both(e, tau, mask)
        if steps and name in ("MAML-AE", "Static-AE"):
            adapt[name] = sc.adaptation_report(models[key], a, sc.to_tensor(sup, DEVICE))
    torch.manual_seed(seed * 1000 + s)
    floor = sc.LSTMAutoencoder(D["normal"].shape[2]).to(DEVICE)
    e, tau, _ = adapted_scores(floor, sup, 0)
    r["LSTM-AE untrained (floor)"] = metrics_both(e, tau, mask)
    e, tau = iforest(sup, s)
    r["Isolation-Forest (K support)"] = metrics_both(e, tau, mask)
    if legacy:
        for name, key in [("Static-AE all-normal (legacy)", "static_all_legacy"),
                          ("MLP-AE all-normal (legacy)", "mlp_all_legacy")]:
            e, tau, _ = adapted_scores(models[key], sup, 0)
            r[name] = metrics_both(e, tau, mask)
        pick = np.random.RandomState(s).choice(len(pool_all), min(CFG["iforest_legacy_windows"], len(pool_all)), replace=False)
        e, tau = iforest(pool_all[pick], s)
        r["Isolation-Forest 2000 normal (legacy)"] = metrics_both(e, tau, mask)
    return r, adapt

def evaluate(models, seed):
    res = {"whole_plant": {}, "heldout_regime": {}, "adaptation": {}}
    for k in CFG["k_shots"]:
        for s in CFG["support_seeds"]:
            sup = pool_all[np.sort(np.random.RandomState([s, k]).choice(len(pool_all), k, replace=False))]
            res["whole_plant"][f"{k}|{s}"], res["adaptation"][f"whole|{k}|{s}"] = score_all(models, sup, s, seed)
            for g, h in heldout.items():
                if not h["testable"] or len(h["pool"]) < k:
                    continue
                sup = h["pool"][np.sort(np.random.RandomState([s, k, g]).choice(len(h["pool"]), k, replace=False))]
                res["heldout_regime"][f"{g}|{k}|{s}"], res["adaptation"][f"heldout|{g}|{k}|{s}"] = \
                    score_all(models, sup, s, seed, mask=h["mask"], legacy=False)
        print(f"  seed {seed}: K={k} scored")
    return res

## 6 — Run all training seeds (finished seeds are skipped)

In [ ]:
for seed in TRAIN_SEEDS:
    if find_all(result_name(seed)):
        print(f"seed {seed}: result exists, skipping"); continue
    t0 = time.time()
    models, info = train_models(seed)
    res = evaluate(models, seed)
    sc.save_json(os.path.join(OUT, result_name(seed)),
                 {"experiment": f"{PLANT} within-plant, corrected", "smoke_test": SMOKE, "train_seed": seed,
                  "config": CFG, "split": sp, "heldout_testable": {str(g): {k: v for k, v in h.items() if k in ("testable", "anomalous", "normal")} for g, h in heldout.items()},
                  "training": info, "results": res, "device": str(DEVICE), "torch": torch.__version__,
                  "wall_seconds": time.time() - t0})
    print(f"seed {seed}: saved {result_name(seed)} ({time.time() - t0:.0f}s)")

## 7 — Summary across training seeds

Values are averaged over support draws within each training seed, then summarised across
training seeds (mean, SD, 95% CI). Paired differences use the training seed as the unit:
two-sided t-test and Wilcoxon test, and a TOST equivalence test for a band of +/- 0.02
ROC-AUC. The original single-run numbers are shown for comparison.

In [ ]:
OLD = {'Static-AE': (0.8402, 0.6933, 4.559), 'MLP-AE': (0.8438, 0.7074, 7.055), 'Isolation-Forest': (0.7983, 0.6871, 1.26), 'MAML-AE 20': (0.817, 0.6612, 6.56), 'MAML-AE 50': (0.8168, 0.6612, 6.526), 'MAML-AE 100': (0.8175, 0.6612, 6.608)}
runs = {s: json.load(open(find_all(result_name(s))[0])) for s in CFG["train_seeds"] if find_all(result_name(s))}
print("training seeds available:", sorted(runs))
METHODS = ["MAML-AE", "MAML-AE (0 steps)", "Static-AE", "Static-AE (0 steps)", "MLP-AE",
           "Static-AE all-normal (legacy)", "MLP-AE all-normal (legacy)", "LSTM-AE untrained (floor)",
           "Isolation-Forest (K support)", "Isolation-Forest 2000 normal (legacy)"]
METRICS = ["roc_auc", "pr_auc", "f1_at_tau", "oracle_best_f1", "separation_ratio"]

def seed_value(run, view, method, k, metric, rule="any", regime=None):
    block = run["results"][view]
    keys = [key for key in block if key.split("|")[-2] == str(k) and (regime is None or key.split("|")[0] == str(regime))]
    vals = [block[key][method][rule].get(metric) for key in keys if method in block[key]]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None

summary = {"whole_plant": {}, "heldout_regime": {}, "paired": {}}
for k in CFG["k_shots"]:
    print(f"\n=== whole-plant view, K = {k} (mean [95% CI] across training seeds; rule 'any') ===")
    print(f"{'method':40s} {'ROC-AUC':>22s} {'F1 at tau':>22s} {'oracle best-F1':>22s} {'separation':>12s}")
    for m in METHODS:
        row = {met: sc.describe_values([seed_value(r, "whole_plant", m, k, met) for r in runs.values()]) for met in METRICS}
        row["roc_auc_half_rule"] = sc.describe_values([seed_value(r, "whole_plant", m, k, "roc_auc", "half") for r in runs.values()])
        summary["whole_plant"][f"{m}|{k}"] = row
        f = lambda e: "n/a" if e["mean"] is None else f"{e['mean']:.4f}" + (f" [{e['ci95'][0]:.3f},{e['ci95'][1]:.3f}]" if "ci95" in e else "")
        print(f"{m:40s} {f(row['roc_auc']):>22s} {f(row['f1_at_tau']):>22s} {f(row['oracle_best_f1']):>22s} {f(row['separation_ratio']):>12s}")
    pairs = [("MAML-AE", "Static-AE"), ("MAML-AE", "MAML-AE (0 steps)"), ("Static-AE", "Static-AE (0 steps)"),
             ("MAML-AE", "Static-AE all-normal (legacy)"), ("MAML-AE", "LSTM-AE untrained (floor)")]
    for a, b in pairs:
        pc = sc.paired_comparison([seed_value(r, "whole_plant", a, k, "roc_auc") for r in runs.values()],
                                  [seed_value(r, "whole_plant", b, k, "roc_auc") for r in runs.values()], CFG["tost_margin"])
        summary["paired"][f"whole|{a} minus {b}|{k}"] = pc
        print(f"  {a} minus {b}: mean {pc['mean']:+.4f}" + (f", 95% CI [{pc['ci95'][0]:+.4f}, {pc['ci95'][1]:+.4f}], "
              f"p(t) {pc['t_test_p_two_sided']:.3f}, TOST p {pc.get('tost_p', float('nan')):.3f}" if "ci95" in pc else ""))
    for g in [g for g, h in heldout.items() if h["testable"]]:
        print(f"\n--- held-out regime {g}, K = {k} ---")
        for m in METHODS[:5] + ["LSTM-AE untrained (floor)", "Isolation-Forest (K support)"]:
            e = sc.describe_values([seed_value(r, "heldout_regime", m, k, "roc_auc", regime=g) for r in runs.values()])
            summary["heldout_regime"][f"{g}|{m}|{k}"] = e
            if e["mean"] is not None:
                print(f"  {m:40s} ROC-AUC {e['mean']:.4f}" + (f" [{e['ci95'][0]:.3f}, {e['ci95'][1]:.3f}]" if "ci95" in e else ""))
        pc = sc.paired_comparison([seed_value(r, "heldout_regime", "MAML-AE", k, "roc_auc", regime=g) for r in runs.values()],
                                  [seed_value(r, "heldout_regime", "Static-AE", k, "roc_auc", regime=g) for r in runs.values()],
                                  CFG["tost_margin"])
        summary["paired"][f"heldout {g}|MAML-AE minus Static-AE|{k}"] = pc

print("\n=== the original single-run numbers (ROC-AUC / oracle best-F1 / separation) ===")
for m, v in OLD.items():
    print(f"  {m:22s} {v}")
sc.save_json(os.path.join(OUT, f"{PLANT}_within_summary{'_SMOKE' if SMOKE else ''}.json"),
             {"smoke_test": SMOKE, "train_seeds_used": sorted(runs), "config": CFG, "summary": summary, "original_numbers": OLD})

## 8 — How to read the results

- A difference between methods counts only if its 95% interval excludes 0.
- "About equal" is supported only if the TOST p-value is below 0.05 (difference shown to lie
  within +/- 0.02 ROC-AUC); otherwise the result is inconclusive.
- MAML minus MAML (0 steps) near 0 means adaptation does not change what is detected.
- The whole-plant view is a sanity baseline; the held-out-regime view is the few-shot test.
- F1 at tau is the deployable F1; oracle best-F1 uses the labels.